# Notebook 4 — Supplementary Analyses for the Revised Manuscript

This notebook answers the reviewer-style checks on the revised experiment (Notebook 3). It re-uses the **exact feature
subsets saved in `results_revised/checkpoint.pkl`**, so every new analysis is tied to the same selections as the main results.

| Part | Check |
|---|---|
| A | Reproducibility: re-train the saved subsets and compare with the stored results |
| B | Re-run the 3-objective search with a full archive: determinism check, convergence (hypervolume), sensitivity to τ and θ |
| C | Extra baselines: random subsets, MI with bootstrap consensus, single-objective genetic algorithm |
| D | Robustness beyond the optimised noise model: uniform, Laplace, quantisation, missing values, meaning-based noise coding |
| E | Full (uncapped) test set, a second classifier (decision tree), per-class precision, false-alarm threshold, single-core latency |
| F | Statistics: Nogueira stability with CIs, equivalence test (TOST), effect sizes, test-set bootstrap CIs |
| G | Direct tests of the IAT leakage hypothesis |

**Inputs:** `cleaned_data/` (Notebook 1), `results_revised/checkpoint.pkl` (Notebook 3) and the raw `Train` folder (Part G).
**Output:** `results_supplementary/` (Excel workbook, summary text, figures). Progress is saved after every run.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import gc, glob, re, time, pickle, platform, warnings
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from pymoo.core.problem import ElementwiseProblem
from pymoo.core.callback import Callback
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.indicators.hv import HV
from pymoo.optimize import minimize
import sklearn, pymoo
warnings.filterwarnings("ignore")

CLEAN_DIR, REV_DIR, RAW_TRAIN = "cleaned_data", "results_revised", "Train"
OUT = os.environ.get("SUPP_OUT", "results_supplementary"); os.makedirs(OUT, exist_ok=True)
N_BOOT = int(os.environ.get("SUPP_BOOT", 1000))
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "font.family": "serif"})

# The checkpoint saved by Notebook 3 (accepts small naming slips such as "checkpoint . pkl" or "checkpoint_2.pkl")
cands = [f for f in glob.glob(os.path.join(REV_DIR, "*")) if f.replace(" ", "").lower().endswith(".pkl") and "checkpoint" in f.lower()]
assert cands, "No checkpoint file found in results_revised/. Copy the 10-run checkpoint from Notebook 3 there as checkpoint.pkl"
CK_FILE = os.path.join(REV_DIR, "checkpoint.pkl") if os.path.join(REV_DIR, "checkpoint.pkl") in cands else sorted(cands)[0]
print("Using checkpoint:", CK_FILE)
with open(CK_FILE, "rb") as fh:
    CK = pickle.load(fh)
assert CK["next_run"] == 10, f"The checkpoint holds {CK['next_run']} runs; the 10-run checkpoint is needed"
(TRAIN_PER_CLASS, TEST_PER_CLASS, NOISE_LEVELS, NOISE_MAIN, NOISE_DRAWS, GA_POP, GA_GEN, GA_PER_CLASS, GA_FOLDS,
 GA_TREES, F1_TOLERANCE, CONSENSUS, FINAL_TREES, FEATURES_ALL, CLASSES) = CK["signature"]
FEATURES_ALL, CLASSES, NOISE_LEVELS = list(FEATURES_ALL), list(CLASSES), list(NOISE_LEVELS)
N_RUNS = int(os.environ.get("SUPP_RUNS", CK["next_run"]))
ENV = (f"{platform.platform()} | CPU: {platform.processor() or 'n/a'} | cores: {os.cpu_count()} | Python {platform.python_version()} | "
       f"scikit-learn {sklearn.__version__} | pymoo {pymoo.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(ENV); print("Runs in checkpoint:", N_RUNS)

Using checkpoint: results_revised\checkpoint.pkl
Windows-10-10.0.26300-SP0 | CPU: Intel64 Family 6 Model 142 Stepping 10, GenuineIntel | cores: 8 | Python 3.11.17 | scikit-learn 1.9.1 | pymoo 0.6.2 | numpy 2.4.6 | pandas 3.0.6
Runs in checkpoint: 10


## Data and helper functions (identical to Notebook 3)

In [2]:
train_all = pd.read_csv(os.path.join(CLEAN_DIR, "train_clean.csv"))
test_all  = pd.read_csv(os.path.join(CLEAN_DIR, "test_clean.csv"))
assert [c for c in train_all.columns if c != "Label"] == FEATURES_ALL
# Guard against incomplete raw files (for example, files still being copied or synced when Notebook 1 ran)
assert len(train_all) == 1713169 and len(test_all) == 375743, (
    f"cleaned_data has {len(train_all):,} train and {len(test_all):,} test rows; expected 1,713,169 and 375,743. "
    "Re-copy the complete raw Train/Test files and re-run Notebook 1 before this notebook.")
train_all[FEATURES_ALL] = train_all[FEATURES_ALL].astype("float32")
test_all[FEATURES_ALL]  = test_all[FEATURES_ALL].astype("float32")
FEATURES = [c for c in FEATURES_ALL if c != "IAT"]
COL = {f: i for i, f in enumerate(FEATURES_ALL)}
SEL_COLS = np.array([COL[f] for f in FEATURES])
INDICATOR = np.array([train_all[c].min() >= 0 and train_all[c].max() <= 1 for c in FEATURES_ALL])

# Meaning-based coding: fixed (never perturbed) = 0/1 flag and protocol indicators + header code fields.
# Protocol Type holds protocol numbers (for example 6 = TCP, 17 = UDP); Duration concentrates at 64, like an IP TTL.
FLAG_OR_PROTO = [c for c in FEATURES_ALL if c.endswith("_flag_number")] + \
    ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv"]
SEMANTIC_FIXED = np.array([c in FLAG_OR_PROTO or c in ("Protocol Type", "Duration") for c in FEATURES_ALL])
print("Range-coded perturbed features:", int((~INDICATOR).sum()), "| meaning-coded perturbed features:", int((~SEMANTIC_FIXED).sum()))
print("Perturbed under range coding but not meaning coding:", [f for f, a, b in zip(FEATURES_ALL, INDICATOR, SEMANTIC_FIXED) if (not a) and b])
print("Perturbed under meaning coding but not range coding:", [f for f, a, b in zip(FEATURES_ALL, INDICATOR, SEMANTIC_FIXED) if a and (not b)])

def sample_per_class(df, label_col, n, seed):
    idx = [g.sample(min(len(g), n), random_state=seed).index for _, g in df.groupby(label_col)]
    return df.loc[np.concatenate(idx)]

def perturb(X, level, rng, fixed):
    Xn = X.copy(); cont = ~fixed
    Xn[:, cont] = np.clip(X[:, cont] * (1 + rng.normal(0, level, (X.shape[0], cont.sum()))), 0, None)
    return Xn.astype("float32")

def make_forest(n_trees, seed, oob=False):
    return RandomForestClassifier(n_estimators=n_trees, class_weight="balanced_subsample", random_state=seed,
                                  n_jobs=-1, oob_score=oob)

test_set = sample_per_class(test_all, "Label", TEST_PER_CLASS, seed=42)
X_test = test_set[FEATURES_ALL].values; y_test = test_set["Label"].values
X_full = test_all[FEATURES_ALL].values;  y_full = test_all["Label"].values
CIDX = {c: i for i, c in enumerate(CLASSES)}
yt_i = np.array([CIDX[c] for c in y_test]); yf_i = np.array([CIDX[c] for c in y_full])
med_train = train_all[FEATURES_ALL].median().values.astype("float32")
del test_all; gc.collect()

def sample_train(seed):
    s = sample_per_class(train_all, "Label", TRAIN_PER_CLASS, seed)
    return s[FEATURES_ALL].values, s["Label"].values

# ----- perturbed test copies (same for every configuration)
cont_r, cont_s = ~INDICATOR, ~SEMANTIC_FIXED
def other_noise(kind, X, rng, fixed, level=0.10):
    Xn = X.copy(); c = ~fixed; shp = (X.shape[0], c.sum())
    if kind == "uniform": e = rng.uniform(-np.sqrt(3) * level, np.sqrt(3) * level, shp)
    elif kind == "laplace": e = rng.laplace(0, level / np.sqrt(2), shp)
    if kind in ("uniform", "laplace"):
        Xn[:, c] = np.clip(X[:, c] * (1 + e), 0, None)
    elif kind == "quantise":       # keep two significant digits of every real-valued measurement
        v = X[:, c].astype("float64"); mag = np.where(v > 0, 10 ** (np.floor(np.log10(np.where(v > 0, v, 1))) - 1), 1)
        Xn[:, c] = np.round(v / mag) * mag
    elif kind.startswith("missing"):  # a share of real-valued entries lost and imputed with the training median
        share = float(kind.split("_")[1]) / 100
        mask = rng.random(shp) < share; sub = Xn[:, c]; sub[mask] = np.broadcast_to(med_train[c], shp)[mask]; Xn[:, c] = sub
    return Xn.astype("float32")

PERT = {}
for lvl in NOISE_LEVELS:   # identical seeds to Notebook 3, used for the reproducibility check
    PERT[f"gauss_{int(lvl*100)}"] = [perturb(X_test, lvl, np.random.default_rng(1000 * d + int(lvl * 100)), INDICATOR) for d in range(NOISE_DRAWS)]
PERT["semantic_gauss_10"] = [perturb(X_test, 0.10, np.random.default_rng(5000 + d), SEMANTIC_FIXED) for d in range(NOISE_DRAWS)]
for kind, base in (("uniform", 6000), ("laplace", 7000), ("missing_10", 8000), ("missing_20", 9000)):
    PERT[kind] = [other_noise(kind, X_test, np.random.default_rng(base + d), SEMANTIC_FIXED) for d in range(NOISE_DRAWS)]
PERT["quantise"] = [other_noise("quantise", X_test, None, SEMANTIC_FIXED)]
print("Perturbation sets:", list(PERT))

Range-coded perturbed features: 20 | meaning-coded perturbed features: 19
Perturbed under range coding but not meaning coding: ['Protocol Type', 'Duration']
Perturbed under meaning coding but not range coding: ['Variance']
Perturbation sets: ['gauss_5', 'gauss_10', 'gauss_20', 'semantic_gauss_10', 'uniform', 'laplace', 'missing_10', 'missing_20', 'quantise']


## Selection functions (identical to Notebook 3, with an archive and a convergence log)

In [3]:
class FeatureSelectionProblem(ElementwiseProblem):
    def __init__(self, folds, n_obj, seed):
        super().__init__(n_var=len(SEL_COLS), n_obj=n_obj, xl=0, xu=1, vtype=bool)
        self.folds, self.seed, self.archive = folds, seed, []
    def _evaluate(self, x, out, *args, **kwargs):
        idx = np.where(x)[0]
        if len(idx) == 0:
            out["F"] = [1.0, 0.0, 1.0][: self.n_obj]; return
        f1s, drops = [], []
        for Xa, ya, Xv, yv, Xvn in self.folds:
            clf = make_forest(GA_TREES, self.seed).fit(Xa[:, idx], ya)
            f1 = f1_score(yv, clf.predict(Xv[:, idx]), average="macro"); f1s.append(f1)
            if self.n_obj == 3:
                drops.append(max(0.0, f1 - f1_score(yv, clf.predict(Xvn[:, idx]), average="macro")))
        F = [1 - np.mean(f1s), len(idx) / self.n_var]
        if self.n_obj == 3: F.append(np.mean(drops))
        F = F[: self.n_obj]
        out["F"] = F
        self.archive.append((np.asarray(x, bool).copy(), np.array(F)))

def make_folds(X_tr, y_tr, seed):
    df = pd.DataFrame(X_tr[:, SEL_COLS]); df["y"] = y_tr
    sub = sample_per_class(df, "y", GA_PER_CLASS, seed)
    Xs, ys = sub.drop(columns="y").values.astype("float32"), sub["y"].values
    ind = INDICATOR[SEL_COLS]; folds = []
    for f, (a, v) in enumerate(StratifiedKFold(GA_FOLDS, shuffle=True, random_state=seed).split(Xs, ys)):
        folds.append((Xs[a], ys[a], Xs[v], ys[v], perturb(Xs[v], NOISE_MAIN, np.random.default_rng(seed * 10 + f), ind)))
    return folds

class HVLog(Callback):
    def __init__(self):
        super().__init__(); self.hv = []; self.ind = HV(ref_point=np.array([1.0, 1.0, 1.0]))
    def notify(self, algorithm):
        self.hv.append(float(self.ind(algorithm.pop.get("F"))))

def run_search(folds, seed, n_obj):
    if n_obj == 1:
        algo = GA(pop_size=GA_POP, sampling=BinaryRandomSampling(), crossover=TwoPointCrossover(),
                  mutation=BitflipMutation(), eliminate_duplicates=True)
        cb = None
    else:
        algo = NSGA2(pop_size=GA_POP, sampling=BinaryRandomSampling(), crossover=TwoPointCrossover(),
                     mutation=BitflipMutation(), eliminate_duplicates=True)
        cb = HVLog() if n_obj == 3 else None
    prob = FeatureSelectionProblem(folds, n_obj, seed)
    kw = {"callback": cb} if cb is not None else {}
    res = minimize(prob, algo, ("n_gen", GA_GEN), seed=seed, verbose=False, **kw)
    archX = np.array([a[0] for a in prob.archive]); archF = np.array([a[1] for a in prob.archive])
    return res, archX, archF, (cb.hv if cb else None)

def pick_consensus(archF, archX, tau, theta):
    val_f1, drop = 1 - archF[:, 0], archF[:, 2]
    good = val_f1 >= val_f1.max() - tau
    good &= drop <= drop[good].min() + tau
    near = np.unique(archX[good], axis=0)
    chosen = np.where(near.mean(axis=0) >= theta)[0]
    if len(chosen) == 0: chosen = np.where(near[0])[0]
    return SEL_COLS[chosen], len(near)

def mi_bootstrap_consensus(X_tr, y_tr, k, seed, B=10):
    freq = np.zeros(len(SEL_COLS)); score = np.zeros(len(SEL_COLS))
    for b in range(B):
        rng = np.random.default_rng(seed * 100 + b)
        idx = rng.choice(len(X_tr), 20000, replace=True)
        s = mutual_info_classif(X_tr[idx][:, SEL_COLS], y_tr[idx], random_state=seed * 100 + b)
        top = np.argsort(s)[::-1][:k]; freq[top] += 1; score += s
    order = np.lexsort((-score, -freq))          # most frequently selected first, ties by mean MI
    return SEL_COLS[order[:k]]
print("Functions ready.")

Functions ready.


## Part A to E — Evaluation loop (one run at a time, progress saved)

In [ ]:
SAVE = os.path.join(OUT, "supp_checkpoint.pkl")
state = {"runs": {}}
if os.path.exists(SAVE):
    with open(SAVE, "rb") as fh: state = pickle.load(fh)
    print("Saved progress found for runs:", sorted(state["runs"]))

sel = pd.DataFrame(CK["selected"])
ORIG = ["All features + IAT (reference)", "All features", "MI (top-k)", "RFE (top-k)", "NSGA-II (2-obj)",
        "NSGA-II (3-obj)", "Proposed: NSGA-II (3-obj) + consensus"]
TAUS, THETAS = [0.005, 0.01, 0.02], [0.3, 0.5, 0.7]

def fast_macro_f1(yi, pi, n=len(CLASSES)):
    cm = np.bincount(yi * n + pi, minlength=n * n).reshape(n, n)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    d = 2 * tp + fp + fn
    return float(np.mean(np.where(d > 0, 2 * tp / np.where(d > 0, d, 1), 0)))

def to_idx(pred): return np.array([CIDX[c] for c in pred])

def evaluate_full(cols, X_tr, y_tr, seed, extended=True):
    cols = list(cols); out = {"n_features": len(cols)}
    m = make_forest(FINAL_TREES, seed, oob=extended).fit(X_tr[:, cols], y_tr)
    p = to_idx(m.predict(X_test[:, cols])); out["macro_F1"] = fast_macro_f1(yt_i, p)
    preds = {"clean": p.astype("int8")}
    for key, sets in PERT.items():
        if not extended and key not in ("gauss_10", "semantic_gauss_10"): continue
        vals = []
        for i, Xn in enumerate(sets):
            q = to_idx(m.predict(Xn[:, cols])); vals.append(fast_macro_f1(yt_i, q))
            if key == "gauss_10" and i == 0: preds["gauss_10"] = q.astype("int8")
        out[f"F1_{key}"] = float(np.mean(vals))
    if extended:
        pf = to_idx(m.predict(X_full[:, cols])); out["F1_full_test"] = fast_macro_f1(yf_i, pf)
        ben = yf_i == CIDX["Benign"]
        out["FPR_full_test"] = float(np.mean(pf[ben] != CIDX["Benign"])); out["FNR_full_test"] = float(np.mean(pf[~ben] == CIDX["Benign"]))
        # per-class precision on the capped test set
        for c, ci in CIDX.items():
            pp = p == ci; out[f"prec_{c}"] = float(np.mean(yt_i[pp] == ci)) if pp.any() else np.nan
        # false-alarm threshold chosen on out-of-bag training predictions (no test data used)
        bi = list(m.classes_).index("Benign"); oob = m.oob_decision_function_
        ytr = np.asarray(y_tr); t = float(np.nanquantile(oob[ytr == "Benign", bi], 0.05))
        prob = m.predict_proba(X_test[:, cols]); att = prob.copy(); att[:, bi] = -1
        pt = np.where(prob[:, bi] >= t, bi, att.argmax(1)); pt = to_idx(m.classes_[pt])
        benT = yt_i == CIDX["Benign"]
        out.update({"thr_t": t, "thr_FPR": float(np.mean(pt[benT] != CIDX["Benign"])),
                    "thr_FNR": float(np.mean(pt[~benT] == CIDX["Benign"])), "thr_macro_F1": fast_macro_f1(yt_i, pt),
                    "FPR": float(np.mean(p[benT] != CIDX["Benign"])), "FNR": float(np.mean(p[~benT] == CIDX["Benign"]))})
        # second classifier: decision tree on the same subset
        dt = DecisionTreeClassifier(class_weight="balanced", random_state=seed).fit(X_tr[:, cols], y_tr)
        out["DT_macro_F1"] = fast_macro_f1(yt_i, to_idx(dt.predict(X_test[:, cols])))
        out["DT_F1_gauss_10"] = float(np.mean([fast_macro_f1(yt_i, to_idx(dt.predict(Xn[:, cols]))) for Xn in PERT["gauss_10"]]))
        out["DT_F1_semantic_gauss_10"] = float(np.mean([fast_macro_f1(yt_i, to_idx(dt.predict(Xn[:, cols]))) for Xn in PERT["semantic_gauss_10"]]))
    del m; gc.collect()
    return out, preds

t0 = time.time()
for run in range(N_RUNS):
    if run in state["runs"]: continue
    seed = run; X_tr, y_tr = sample_train(seed); R = {"rows": [], "preds": {}, "sens": [], "hv": None}
    subsets = {m: np.array([COL[f] for f in sel[(sel.run == run) & (sel.method == m)].features.iloc[0]]) for m in ORIG}
    k = len(subsets["Proposed: NSGA-II (3-obj) + consensus"])
    folds = make_folds(X_tr, y_tr, seed)

    # Part B: re-run the 3-objective search with archive and hypervolume log
    t1 = time.perf_counter(); res3, aX, aF, hv = run_search(folds, seed, 3); R["search3_s"] = time.perf_counter() - t1
    selP, n_near = pick_consensus(aF, aX, F1_TOLERANCE, CONSENSUS)
    R["reproduced"] = set(FEATURES_ALL[i] for i in selP) == set(FEATURES_ALL[i] for i in subsets["Proposed: NSGA-II (3-obj) + consensus"])
    R["hv"] = hv; R["n_archive"] = len(aX)
    if not R["reproduced"]:
        raise RuntimeError(
            f"Run {run + 1}: the proposed subset was NOT reproduced, so this Python environment builds different models "
            "from the one used for Notebook 3. Nothing was saved for this run. Delete results_supplementary/, switch to the "
            "'Python (iomt)' kernel created by Notebook 0 (Python 3.11, current scikit-learn and pymoo) and run again.")
    # Part C: extra baselines
    rng = np.random.default_rng(1000 + run)
    subsets["Random (k)"] = rng.choice(SEL_COLS, size=k, replace=False)
    subsets["MI bootstrap consensus (k)"] = mi_bootstrap_consensus(X_tr, y_tr, k, seed)
    t1 = time.perf_counter(); res1, _, _, _ = run_search(folds, seed, 1); R["search1_s"] = time.perf_counter() - t1
    subsets["GA single-objective"] = SEL_COLS[np.where(np.atleast_2d(res1.X)[0])[0]]
    R["subsets"] = {m: [FEATURES_ALL[i] for i in v] for m, v in subsets.items()}

    for m, cols in subsets.items():
        res, preds = evaluate_full(cols, X_tr, y_tr, seed, extended=True)
        R["rows"].append({"run": run, "method": m, **res})
        if m in ORIG: R["preds"][m] = preds
    # sensitivity of tau and theta (clean and two noise conditions)
    for tau in TAUS:
        for theta in THETAS:
            cols, nn = pick_consensus(aF, aX, tau, theta)
            res, _ = evaluate_full(cols, X_tr, y_tr, seed, extended=False)
            R["sens"].append({"run": run, "tau": tau, "theta": theta, "n_near": nn, "features": [FEATURES_ALL[i] for i in cols], **res})
    # Part E: single-core latency (run 1 only)
    if run == 0:
        lat = {}
        for m in ["All features", "MI (top-k)", "RFE (top-k)", "Proposed: NSGA-II (3-obj) + consensus"]:
            cols = list(subsets[m]); mm = make_forest(FINAL_TREES, seed).fit(X_tr[:, cols], y_tr); mm.set_params(n_jobs=1)
            rows_ = X_test[:300, cols]; ts = []
            for i in range(len(rows_)):
                a = time.perf_counter(); mm.predict(rows_[i:i+1]); ts.append(time.perf_counter() - a)
            a = time.perf_counter(); mm.predict(X_test[:, cols]); bt = time.perf_counter() - a
            lat[m] = {"median_ms_per_record": 1000 * float(np.median(ts)), "batch_ms_per_1000_1core": 1000 * bt / len(X_test) * 1000}
            del mm
        R["latency"] = lat
    state["runs"][run] = R
    with open(SAVE, "wb") as fh: pickle.dump(state, fh)
    del X_tr, y_tr, folds; gc.collect()
    print(f"run {run+1}/{N_RUNS} done | reproduced proposed subset: {R['reproduced']} | elapsed {(time.time()-t0)/60:.1f} min", flush=True)

Saved progress found for runs: [0, 1, 2, 3]
run 5/10 done | reproduced proposed subset: True | elapsed 15.1 min
run 6/10 done | reproduced proposed subset: True | elapsed 32.0 min
run 7/10 done | reproduced proposed subset: True | elapsed 49.6 min
run 8/10 done | reproduced proposed subset: True | elapsed 66.3 min
run 9/10 done | reproduced proposed subset: True | elapsed 82.0 min


C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\paralle

## Part F — Statistics

In [ ]:
rows = pd.DataFrame([r for R in state["runs"].values() for r in R["rows"]])
ck_rows = pd.DataFrame(CK["rows"])
# A. reproducibility of the stored results
rep = rows[rows.method.isin(ORIG)].merge(ck_rows[["run", "method", "macro_F1", "F1_noise_10"]], on=["run", "method"], suffixes=("_re", "_ck"))
rep["dF1"] = (rep.macro_F1_re - rep.macro_F1_ck).abs(); rep["dN"] = (rep.F1_gauss_10 - rep.F1_noise_10).abs()
repro = {"max_abs_diff_clean_F1": rep.dF1.max(), "max_abs_diff_F1_noise10": rep.dN.max(),
         "proposed_subset_reproduced_runs": int(sum(R["reproduced"] for R in state["runs"].values())), "runs": len(state["runs"])}
print(repro)

METHODS = ORIG + ["Random (k)", "MI bootstrap consensus (k)", "GA single-objective"]
def ci95(x):
    x = np.asarray(x, float); return 0.0 if len(x) < 2 else stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
MET = ["n_features", "macro_F1", "F1_gauss_10", "F1_semantic_gauss_10", "F1_uniform", "F1_laplace", "F1_quantise",
       "F1_missing_10", "F1_missing_20", "F1_full_test", "FPR_full_test", "FNR_full_test", "FPR", "FNR", "thr_FPR", "thr_FNR",
       "thr_macro_F1", "DT_macro_F1", "DT_F1_gauss_10", "DT_F1_semantic_gauss_10"] + [f"prec_{c}" for c in CLASSES]
mean_s = rows.groupby("method")[MET].mean().reindex(METHODS); ci_s = rows.groupby("method")[MET].agg(ci95).reindex(METHODS)

# Nogueira stability with its asymptotic 95% CI (variance estimator of Nogueira, Sechidis and Brown, 2018).
# A bootstrap over runs is not used because duplicated runs agree with themselves and inflate phi.
dsel = len(FEATURES)
def nogueira(sets, with_var=False):
    Z = np.array([[f in s for f in FEATURES] for s in sets], float); M = len(sets)
    p = Z.mean(0); kb = Z.sum(1).mean(); k = Z.sum(1); den = (kb / dsel) * (1 - kb / dsel)
    phi = 1 - (M / (M - 1)) * (p * (1 - p)).mean() / den
    if not with_var: return phi
    ph = np.array([((Z[i] * p).mean() - k[i] * kb / dsel**2 + (phi / 2) * (2 * k[i] * kb / dsel**2 - k[i] / dsel - kb / dsel + 1)) / den
                   for i in range(M)])
    return phi, (4 / M**2) * ((ph - ph.mean())**2).sum()
def jacc(sets): return np.mean([len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i+1:]])
stab = []
for m in ["MI (top-k)", "RFE (top-k)", "NSGA-II (2-obj)", "NSGA-II (3-obj)", "Proposed: NSGA-II (3-obj) + consensus",
          "Random (k)", "MI bootstrap consensus (k)", "GA single-objective"]:
    sets = [set(state["runs"][r]["subsets"][m]) for r in sorted(state["runs"])]
    phi, var = nogueira(sets, with_var=True); half = 1.959964 * np.sqrt(var)
    k_ = np.mean([len(s) for s in sets]); jr = (k_ * k_ / dsel) / (2 * k_ - k_ * k_ / dsel)
    stab.append({"method": m, "mean k": k_, "Jaccard": jacc(sets), "Jaccard expected by chance": jr,
                 "Nogueira phi": phi, "phi CI low": phi - half, "phi CI high": phi + half})
stability = pd.DataFrame(stab)

# Effect sizes (matched-pairs rank-biserial) and Holm across all comparisons and metrics, from the stored main results
P = "Proposed: NSGA-II (3-obj) + consensus"
def rank_biserial(d):
    d = np.asarray(d); d = d[d != 0]
    if len(d) == 0: return 0.0
    r = stats.rankdata(np.abs(d)); return float((r[d > 0].sum() - r[d < 0].sum()) / r.sum())
eff = []
for metric in ["macro_F1", "F1_noise_10", "F1_drop_10", "n_features", "infer_ms_per_1000", "train_time_s"]:
    wide = ck_rows.pivot(index="run", columns="method", values=metric)
    for o in [m for m in ORIG if m != P]:
        d = wide[P] - wide[o]
        p = 1.0 if np.allclose(d, 0) else stats.wilcoxon(wide[P], wide[o]).pvalue
        eff.append({"metric": metric, "proposed vs": o, "mean difference": d.mean(), "rank-biserial r": rank_biserial(d), "Wilcoxon p": p})
eff = pd.DataFrame(eff)
def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run_ = 0
    for i, j in enumerate(o): run_ = max(run_, min(1, (len(p) - i) * p[j])); adj[j] = run_
    return adj
eff["Holm p (within metric)"] = eff.groupby("metric")["Wilcoxon p"].transform(lambda s: holm(s.values))
eff["Holm p (all 36 tests)"] = holm(eff["Wilcoxon p"].values)

# Extended paired comparisons: proposed vs every other configuration (including the new baselines) on the
# out-of-family perturbations and the decision tree, from the supplementary runs. Holm within each metric (9 tests).
eff_ext = []
for metric in ["macro_F1", "F1_gauss_10", "F1_semantic_gauss_10", "F1_uniform", "F1_laplace", "F1_quantise",
               "F1_missing_10", "F1_missing_20", "DT_macro_F1", "DT_F1_gauss_10", "DT_F1_semantic_gauss_10"]:
    wide = rows.pivot(index="run", columns="method", values=metric)
    for o in [m for m in METHODS if m != P]:
        d = wide[P] - wide[o]
        p = 1.0 if np.allclose(d, 0) else stats.wilcoxon(wide[P], wide[o]).pvalue
        eff_ext.append({"metric": metric, "proposed vs": o, "mean difference": d.mean(), "runs favouring proposed": int((d > 0).sum()),
                        "rank-biserial r": rank_biserial(d), "Wilcoxon p": p})
eff_ext = pd.DataFrame(eff_ext)
eff_ext["Holm p (within metric)"] = eff_ext.groupby("metric")["Wilcoxon p"].transform(lambda s: holm(s.values))

# Equivalence test (TOST) for clean macro-F1, proposed vs all features, margin 0.01
w = ck_rows.pivot(index="run", columns="method", values="macro_F1"); d = (w[P] - w["All features"]).values
delta = 0.01
p_low = stats.wilcoxon(d + delta, alternative="greater").pvalue; p_up = stats.wilcoxon(d - delta, alternative="less").pvalue
tt = stats.t.ppf(0.95, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
tost = {"mean difference": d.mean(), "90% CI low": d.mean() - tt, "90% CI high": d.mean() + tt, "margin": delta,
        "Wilcoxon TOST p": max(p_low, p_up), "equivalent at 0.05": max(p_low, p_up) < 0.05}
print(tost)

# Test-set bootstrap CIs (records resampled; mean over runs), and paired difference proposed minus all features
rng = np.random.default_rng(1); n = len(yt_i); boot = {m: [] for m in ORIG}; bdiff = []; bdiffN = []
runs_ = sorted(state["runs"])
for _ in range(N_BOOT):
    ii = rng.integers(0, n, n); vals = {}
    for m in ORIG:
        vals[m] = np.mean([fast_macro_f1(yt_i[ii], state["runs"][r]["preds"][m]["clean"][ii]) for r in runs_])
        boot[m].append(vals[m])
    bdiff.append(vals[P] - vals["All features"])
    bdiffN.append(np.mean([fast_macro_f1(yt_i[ii], state["runs"][r]["preds"][P]["gauss_10"][ii]) -
                           fast_macro_f1(yt_i[ii], state["runs"][r]["preds"]["All features"]["gauss_10"][ii]) for r in runs_]))
bootci = pd.DataFrame({m: {"macro_F1 2.5%": np.percentile(v, 2.5), "macro_F1 97.5%": np.percentile(v, 97.5)} for m, v in boot.items()}).T
bootdiff = {"clean F1 diff (proposed - all) 2.5%": np.percentile(bdiff, 2.5), "97.5%": np.percentile(bdiff, 97.5),
            "F1 at 10% noise diff 2.5%": np.percentile(bdiffN, 2.5), "F1 at 10% noise diff 97.5%": np.percentile(bdiffN, 97.5)}
print(bootdiff)

sens = pd.DataFrame([s for R in state["runs"].values() for s in R["sens"]])
sens["k"] = sens["features"].apply(len)
sens_tab = sens.groupby(["tau", "theta"])[["k", "n_near", "macro_F1", "F1_gauss_10", "F1_semantic_gauss_10"]].mean()
sens_stab = sens.groupby(["tau", "theta"])["features"].apply(lambda s: nogueira([set(x) for x in s]))
sens_tab["Nogueira phi"] = sens_stab
hv = np.array([state["runs"][r]["hv"] for r in runs_])
lat = pd.DataFrame(state["runs"][0]["latency"]).T
print(sens_tab.round(4)); print(lat.round(3))

## Part G — Direct tests of the IAT leakage hypothesis

In [6]:
# G1. Can IAT identify the capture file (part 1 to 4) within the same attack class? Chance level is 25%.
parts = []
for f in sorted(glob.glob(os.path.join(RAW_TRAIN, "TCP_IP-D*oS-SYN[1-4]_train.pcap.csv"))):
    m_ = re.search(r"(TCP_IP-D?DoS-SYN)(\d)_train", os.path.basename(f))
    df = pd.read_csv(f).sample(20000, random_state=0); df["cls"] = m_.group(1); df["part"] = int(m_.group(2)); parts.append(df)
P4 = pd.concat(parts, ignore_index=True)
others = [c for c in FEATURES_ALL if c != "IAT"]
g1 = []
for cls, g in P4.groupby("cls"):
    for name, cols in (("IAT only", ["IAT"]), ("All features except IAT", others)):
        pred = cross_val_predict(RandomForestClassifier(100, random_state=0, n_jobs=-1), g[cols].values, g["part"].values,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=0))
        g1.append({"class": cls, "predictors": name, "accuracy predicting capture file": float(np.mean(pred == g["part"].values))})
g1 = pd.DataFrame(g1); print(g1)

# G2. IAT on its own as an attack classifier (run 1 training sample, fixed test set)
X_tr, y_tr = sample_train(0); ci = [COL["IAT"]]
mI = make_forest(FINAL_TREES, 0).fit(X_tr[:, ci], y_tr)
g2 = {"IAT-only macro-F1 (clean)": fast_macro_f1(yt_i, to_idx(mI.predict(X_test[:, ci]))),
      "IAT-only macro-F1 (5% noise)": fast_macro_f1(yt_i, to_idx(mI.predict(PERT["gauss_5"][0][:, ci])))}
print(g2)

# G3. Per-class IAT location and spread in training and test data
tr_iat = train_all.groupby("Label")["IAT"].describe()[["50%", "25%", "75%"]]
te_iat = pd.DataFrame({"IAT": X_test[:, COL["IAT"]], "Label": y_test}).groupby("Label")["IAT"].describe()[["50%", "25%", "75%"]]
g3 = tr_iat.join(te_iat, lsuffix=" train", rsuffix=" test"); print(g3)

C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\paralle

             class               predictors  accuracy predicting capture file
0  TCP_IP-DDoS-SYN                 IAT only                          0.972187
1  TCP_IP-DDoS-SYN  All features except IAT                          0.719512
2   TCP_IP-DoS-SYN                 IAT only                          0.970413
3   TCP_IP-DoS-SYN  All features except IAT                          0.685187


C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\Home\anaconda3\envs\iomt\Lib\site-packages\sklearn\utils\paralle

{'IAT-only macro-F1 (clean)': 0.7533474047656916, 'IAT-only macro-F1 (5% noise)': 0.24809618291570126}
                           50% train     25% train    75% train      50% test  \
Label                                                                           
ARP_Spoofing             169402128.0  4.361248e-03  169402416.0  8.470138e+07   
Benign                   169461488.0  1.293430e-02  169465088.0  1.006676e-01   
MQTT-DDoS-Connect_Flood   84667032.0  8.466702e+07   84667040.0  8.466705e+07   
MQTT-Malformed_Data      169323360.0  2.990007e-04  169323376.0  1.693234e+08   
Recon-Port_Scan          169410144.0  1.063108e-04  169410240.0  1.694103e+08   
Recon-VulScan            169453248.0  2.494407e-03  169453280.0  1.694533e+08   
TCP_IP-DDoS-SYN           84696328.0  8.469630e+07   84696408.0  8.469645e+07   
TCP_IP-DoS-SYN            84674632.0  8.467461e+07   84674664.0  8.467474e+07   

                             25% test     75% test  
Label                            

## Save results and figures

In [7]:
with pd.ExcelWriter(os.path.join(OUT, "supplementary_results.xlsx")) as xw:
    pd.Series(repro).to_frame("value").to_excel(xw, sheet_name="Reproducibility")
    mean_s.T.to_excel(xw, sheet_name="Means"); ci_s.T.to_excel(xw, sheet_name="CI95")
    stability.to_excel(xw, sheet_name="Stability", index=False)
    eff.to_excel(xw, sheet_name="Effect_sizes", index=False)
    eff_ext.to_excel(xw, sheet_name="Effect_sizes_extended", index=False)
    pd.Series(tost).to_frame("value").to_excel(xw, sheet_name="TOST")
    bootci.to_excel(xw, sheet_name="Test_bootstrap"); pd.Series(bootdiff).to_frame("value").to_excel(xw, sheet_name="Bootstrap_diff")
    sens_tab.to_excel(xw, sheet_name="Sensitivity")
    pd.DataFrame(hv, columns=[f"gen{i+1}" for i in range(hv.shape[1])]).to_excel(xw, sheet_name="Hypervolume")
    lat.to_excel(xw, sheet_name="Latency_1core")
    g1.to_excel(xw, sheet_name="IAT_capture_file", index=False); pd.Series(g2).to_frame("value").to_excel(xw, sheet_name="IAT_only")
    g3.to_excel(xw, sheet_name="IAT_by_class"); rows.to_excel(xw, sheet_name="All_runs", index=False)

pd.set_option("display.width", 250)
text = "\n".join(["SUPPLEMENTARY ANALYSES", "=" * 60, "Environment: " + ENV,
    "\nREPRODUCIBILITY:\n" + pd.Series(repro).to_string(),
    "\nMEANS (rows = metrics):\n" + mean_s.T.round(4).to_string(),
    "\nCI95:\n" + ci_s.T.round(4).to_string(),
    "\nSTABILITY:\n" + stability.round(4).to_string(index=False),
    "\nEFFECT SIZES:\n" + eff.round(5).to_string(index=False),
    "\nEFFECT SIZES (EXTENDED):\n" + eff_ext.round(5).to_string(index=False),
    "\nTOST:\n" + pd.Series(tost).to_string(),
    "\nTEST BOOTSTRAP CI:\n" + bootci.round(4).to_string() + "\n" + pd.Series(bootdiff).round(4).to_string(),
    "\nSENSITIVITY:\n" + sens_tab.round(4).to_string(),
    "\nHYPERVOLUME mean by generation:\n" + pd.Series(hv.mean(0)).round(4).to_string(),
    "\nLATENCY (1 core, run 1):\n" + lat.round(4).to_string(),
    "\nIAT G1:\n" + g1.round(4).to_string(index=False), "\nIAT G2:\n" + pd.Series(g2).round(4).to_string(),
    "\nIAT G3:\n" + g3.to_string()])
with open(os.path.join(OUT, "supplementary_summary.txt"), "w", encoding="utf-8") as fh: fh.write(text)
print(text[:3000])

SUPPLEMENTARY ANALYSES
Environment: Windows-10-10.0.26300-SP0 | CPU: Intel64 Family 6 Model 142 Stepping 10, GenuineIntel | cores: 8 | Python 3.11.17 | scikit-learn 1.9.1 | pymoo 0.6.2 | numpy 2.4.6 | pandas 3.0.6

REPRODUCIBILITY:
max_abs_diff_clean_F1               0.0
max_abs_diff_F1_noise10             0.0
proposed_subset_reproduced_runs    10.0
runs                               10.0

MEANS (rows = metrics):
method                        All features + IAT (reference)  All features  MI (top-k)  RFE (top-k)  NSGA-II (2-obj)  NSGA-II (3-obj)  Proposed: NSGA-II (3-obj) + consensus  Random (k)  MI bootstrap consensus (k)  GA single-objective
n_features                                           42.0000       41.0000     20.5000      20.5000           9.0000          13.6000                                20.5000     20.5000                     20.5000              21.1000
macro_F1                                              0.8784        0.7945      0.7880       0.7935           0.769